# In Class Assignment

**Name:** Christine Wu
**Date:** 04/07/26

## Submission Instructions
* Submit a link to your Jupyter Notebook in your GitHub repository. Make sure your settings will allow viewing of your notebook.

* Your notebook must be fully run (all outputs visible) and committed and pushed to your repository before the deadline.



## Activity

### 1. (3 points) Using pandas, load the dataset and display the first 5 rows.


In [40]:
!pip install sweetviz

In [41]:
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression



In [42]:
# loading the dataset
insurance = pd.read_csv('/Users/christinewu/Downloads/insurance.csv')
insurance.head(5)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


### 2. (3 points) Using SweetViz, do a quick EDA of the data set. In a markdown cell describe the main trends and patterns you see in the data.


In [43]:
# Initial EDA with Sweetviz
report = sv.analyze(insurance)
report.show_html('insurance_report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)

Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


The main trends and patterns I see in the data are that most respondents were 18 or 19 years old, there is an equal distribution of male and female respondents, the average BMI is 30.7, most people do not have children, 20% of the respondents are smokers, there is an approximately equal distribution of regions across respondents, and the majority of respondents were charged less than 20k.

### 3. (4 points) Define the input variables (X) and target variable (y). Split the data into training (80%) and testing (20%).

In [44]:
# Encoding categorical variables
insurance_encoded = pd.get_dummies(insurance, drop_first=True).astype(int)
insurance_encoded.head

<bound method NDFrame.head of       age  bmi  ...  region_southeast  region_southwest
0      19   27  ...                 0                 1
1      18   33  ...                 1                 0
2      28   33  ...                 1                 0
3      33   22  ...                 0                 0
4      32   28  ...                 0                 0
...   ...  ...  ...               ...               ...
1333   50   30  ...                 0                 0
1334   18   31  ...                 0                 0
1335   18   36  ...                 1                 0
1336   21   25  ...                 0                 1
1337   61   29  ...                 0                 0

[1338 rows x 9 columns]>

In [45]:
# Define X and y variables
X = insurance_encoded.drop('charges', axis = 1)
y = insurance_encoded['charges']

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)


### 4. (4 points) Scale the input variables.

In [46]:
# Scaling the input variables
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 5. (5 points) Build a baseline linear regressor and random forest regressor. Report both R² and MSE for both models. Add a markdown cell discussing the performance of each of the models on the test data. 

In [47]:
# Baseline linear regression model
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

# Baseline random forest model
baseline_rf = RandomForestRegressor(random_state = 42)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

# Report both R² and MSE for both models
print("Linear Regression Performance:")
print(f"MSE: {mse_lr:.2f}")
print(f"R²: {r2_lr:.4f}")

print("\nRandom Forest Performance:")
print(f"MSE: {mse_rf:.2f}")
print(f"R²: {r2_rf:.4f}")

Linear Regression Performance:
MSE: 33566439.74
R²: 0.7838

Random Forest Performance:
MSE: 22179121.67
R²: 0.8571


The random forest model has a better R² because it yields a higher value, which indicates that more of the variance in the target variable can be explained by the model. The random forest model also has a better MSE value because it has a lower value, indicating that there is less error between the predicted and actual values (on average).

### 6. (5 points) Use GridSearchCV to tune n_estimators, max_depth, min_samples_split and max_features. What is the best set of hyperparameters from your search?



In [48]:
# Define parameter grid
param_grid = {
    'n_estimators': [10, 150, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}

# Initialize base model
rf = RandomForestRegressor(random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

# Fit on training data
grid_search.fit(X_train, y_train)

# Best model
best_rf = grid_search.best_estimator_

# Predictions with tuned model
y_pred_best_rf = best_rf.predict(X_test)

# Metrics
mse_best_rf = mean_squared_error(y_test, y_pred_best_rf)
r2_best_rf = r2_score(y_test, y_pred_best_rf)

# Best Hyperparameters
print("Best Hyperparameters:")
print(grid_search.best_params_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Hyperparameters:
{'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 150}


### 7. (5 points) Examine the top 20 models (based on rank) from your GridSearchCV results. Add a markdown cell and discuss which set of hyperparameters you would choose and why.

In [49]:
# Convert results to DataFrame
cv_results = pd.DataFrame(grid_search.cv_results_)

# Sort by rank and select top 20
top_20 = cv_results.sort_values(by='rank_test_score').head(20)

# Select columns for viewing
top_20 = top_20[[
    'rank_test_score',
    'mean_test_score',
    'std_test_score',
    'param_n_estimators',
    'param_max_depth',
    'param_min_samples_split',
    'param_max_features'
]]

top_20

,rank_test_score,mean_test_score,std_test_score,param_n_estimators,param_max_depth,param_min_samples_split,param_max_features
40,1,0.841800,0.039713,150,10,5,log2
41,2,0.841638,0.041278,300,10,5,log2
68,3,0.841585,0.040260,300,20,5,log2
95,3,0.841585,0.040260,300,30,5,log2
14,3,0.841585,0.040260,300,None,5,log2
16,6,0.841426,0.039579,150,None,10,log2
97,6,0.841426,0.039579,150,30,10,log2
70,6,0.841426,0.039579,150,20,10,log2
94,9,0.841085,0.039655,150,30,5,log2
13,9,0.841085,0.039655,150,None,5,log2


I would choose the hyperparameters from my top performing model, which has these hyperparameters:

best_params = {
    'n_estimators': 150,
    'max_depth': 10,
    'min_samples_split': 5,
    'max_features': 'log2'}

I would choose these because this combination achieved the highest cross-validated R squared value while staying at a lower complexity. The differences in the 2nd through 5th ranked models only add complexity and there isn't much difference in the R squared value.

### 8. (3 points--if time) Compare the training versus test performance using a Random Forest model tuned with the best parameters from GridSearch CV. Is your model overfitting and how do you know?

In [50]:
# Best parameters from GridSearchCV
best_params = {
    'n_estimators': 150,
    'max_depth': 10,
    'min_samples_split': 5,
    'max_features': 'log2',
    'random_state': 42
}

# Train final model
final_rf = RandomForestRegressor(**best_params)
final_rf.fit(X_train, y_train)

# Predictions
y_train_pred = final_rf.predict(X_train)
y_test_pred = final_rf.predict(X_test)

# Metrics - Training
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Metrics - Test
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print comparison
print(f"{'Metric':<15}{'Train':<15}{'Test':<15}")
print(f"{'MSE':<15}{train_mse:<15.2f}{test_mse:<15.2f}")
print(f"{'R²':<15}{train_r2:<15.4f}{test_r2:<15.4f}")

Metric         Train          Test           
MSE            10024018.97    20328418.32    
R²             0.9305         0.8691         


Yes, my model is overfitting. I know that it is overfitting because the train R-squared value is greater than the test R-squared value, suggesting that the model learned the training data noise and does not generalize as well to the test data. 